In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
llm_name: str = "Qwen/Qwen3-0.6B"
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    trust_remote_code=True,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(llm_name)

c:\Users\Charlie\anaconda3\envs\ai_assistant\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0106 07:52:34.627000 41488 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
lora_r = 16
lora_alpha = 32
lora_dropout = 0.1

# llm = get_peft_model(llm, LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=lora_r,
#     lora_alpha=lora_alpha,
#     lora_dropout=lora_dropout,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     inference_mode=False,
# ))
llm = PeftModel.from_pretrained(llm, "checkpoints/base_llm_qwen3-0.6b_lora/final")
llm.print_trainable_parameters()

trainable params: 0 || all params: 606,142,464 || trainable%: 0.0000


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel

num_vectors = 32

class PercieverToQwenCrossAttentionModel(nn.Module):
    """
    Model that takes input text, encodes it with an embedding model,
    and projects it to 32 vectors of dimension equal to Qwen's hidden_dim
    using a cross-attention mechanism.
    """
    def __init__(self, 
                 perceiver_model, 
                 perceiver_dim: int,
                 qwen_hidden_dim: int, 
                 num_vectors: int = 32):
        super().__init__()
        self.perceiver_model = perceiver_model
        self.qwen_hidden_dim = qwen_hidden_dim
        self.num_vectors = num_vectors

        # Linear to project embedding model's output to a "memory" for cross-attention
        self.memory_proj = nn.Linear(
            perceiver_dim, qwen_hidden_dim
        )

        # Learnable queries to use for cross-attention
        self.queries = nn.Parameter(torch.randn(1, num_vectors, qwen_hidden_dim))

        # Cross-attention module: query = [num_vectors, hidden_dim], key & value = [seq, hidden_dim]
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=qwen_hidden_dim,
            num_heads=4,
            batch_first=True
        )

    def freeze_projection(self):
        for param in self.memory_proj.parameters():
            param.requires_grad = False

        for param in self.cross_attn.parameters():
            param.requires_grad = False

        self.queries.requires_grad = False
        

    def forward(self, input_ids=None, attention_mask=None):
        """
        text: list[str], or tensor as accepted by embedding_model
        Returns: projected_vectors: shape (batch, num_vectors, qwen_hidden_dim)
        """
        embed_layer = llm.get_input_embeddings()
        embed_tokens = embed_layer(input_ids)
        perceiver_output = self.perceiver_model(embed_tokens)

        # 2. Project embedding to Qwen hidden dimension
        memory = self.memory_proj(perceiver_output)

        # 3. Prepare queries: expand learnable [1, num_vectors, h] to batch
        queries = self.queries.expand(perceiver_output.shape[0], -1, -1)  # (batch, num_vectors, dim)

        # 4. Cross attention: query=(B, N, D), key/value=(B, 1, D)
        # nn.MultiheadAttention expects shape (batch_size, seq_length, embed_dim)
        attended, _ = self.cross_attn(queries, memory, memory)  # (batch, num_vectors, dim)
        return attended

    def load_projector(self, path):
        import os

        additional_path = os.path.join(path, "additional_components.pt")
        additional_components = torch.load(additional_path, map_location=self.device)
        self.memory_proj.load_state_dict(additional_components["memory_proj"])
        self.queries.load_state_dict(additional_components["queries"])
        self.cross_attn.load_state_dict(additional_components["cross_attn"])


from perceiver_module import PerceiverIOModule
perceiver_model = PerceiverIOModule(
    input_dim=llm.config.hidden_size,
).to(device)

cross_proj = PercieverToQwenCrossAttentionModel(
    perceiver_model=perceiver_model,
    perceiver_dim=perceiver_model.perceiver.config.d_latents,
    qwen_hidden_dim=llm.config.hidden_size,
    num_vectors=num_vectors
).to(device)
cross_proj.freeze_projection()
class QwenWithCrossAttention(nn.Module):
    def __init__(self, qwen_model, cross_proj):
        super().__init__()
        self.qwen_model = qwen_model
        self.cross_proj = cross_proj

    def forward(self, input_ids=None, attention_mask=None, labels=None, text=None):
        # Encode text with embedding model
        embeddings = self.cross_proj(input_ids, attention_mask)
        embed_layer = self.qwen_model.get_input_embeddings()
        text_embeddings = embed_layer(input_ids)
        total_embeddings = torch.cat([text_embeddings, embeddings], dim=1)
        
        # Pass embeddings to Qwen model
        outputs = self.qwen_model(inputs_embeds=total_embeddings, attention_mask=attention_mask, labels=labels)
        
        return outputs

    def get_num_trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

model = QwenWithCrossAttention(llm, cross_proj)
print("Trainable Parameters: ", model.get_num_trainable_parameters())

Perceiver latent dimension: 512
Perceiver number of latents: 784
Created input projection: 1024 -> 704
Created CrossAttentionCompressor: 784 latents -> 1 vector(s)
Trainable Parameters:  17041536


In [5]:
test = "Hello"
input_ids = tokenizer.encode(test, return_tensors="pt").to(device)
output = cross_proj(input_ids)
print(output, output.shape)

tensor([[[ 6.3068,  6.0932,  0.9804,  ..., -2.3626, -3.0060, -1.1886],
         [ 6.3068,  6.0932,  0.9804,  ..., -2.3626, -3.0060, -1.1886],
         [ 6.3068,  6.0932,  0.9804,  ..., -2.3626, -3.0060, -1.1886],
         ...,
         [ 6.3068,  6.0932,  0.9804,  ..., -2.3626, -3.0060, -1.1886],
         [ 6.3068,  6.0932,  0.9804,  ..., -2.3626, -3.0060, -1.1886],
         [ 6.3068,  6.0932,  0.9804,  ..., -2.3626, -3.0060, -1.1886]]],
       device='cuda:0', grad_fn=<TransposeBackward0>) torch.Size([1, 32, 1024])


In [6]:
from datasets import load_dataset
from data_utils import get_clean_turns
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from typing import Optional
import torch

data_path = "Bossologist/reddit-conversations-processed"

class ConversationDataset(Dataset):
    """
    Dataset for conversation data with turn boundaries.
    Supports both local JSON files and HuggingFace datasets.
    """
    def __init__(
        self,
        data_path: str,
        tokenizer,
        max_length: int = 128,
    ):
        """
        Initialize dataset.
        
        Args:
            data_path: Path to JSON file with conversations OR HuggingFace dataset name
            tokenizer: Tokenizer to use
            max_length: Maximum sequence length
            turn_separator: Token/string used to separate turns
            text_column: Column name in HuggingFace dataset containing conversation text
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        print(f"Loading dataset from HuggingFace: {data_path}")
        hf_dataset = load_dataset(data_path)
        dataset_split = hf_dataset['train']
        
        # Convert to list of dicts
        self.data = []
        # Auto-detect text column if not specified
        possible_columns = ['text', 'conversation', 'input', 'content', 'prompt', 'messages']
        text_column = next(
            (col for col in possible_columns if col in dataset_split.column_names),
            dataset_split.column_names[0] if dataset_split.column_names else None
        )
        for item in dataset_split:
            text = item.get(text_column)    
            turns = get_clean_turns(text)
            for i in range(len(turns)):
                if turns[i]["role"] != "system":
                    self.data.append({"conversation": turns[i]['content']})
        
        print(f"Loaded {len(self.data)} examples from HuggingFace dataset")
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        """
        Get a single conversation example.
        
        Expected data format:
        {
            "conversation": "turn1<|turn|>turn2<|turn|>turn3",
            "next_token": "token"  # Optional, for supervised learning
        }
        """
        item = self.data[idx]
        conversation = item["conversation"]
        
        # Tokenize full conversation
        encoded = self.tokenizer(
            conversation,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        
        token_ids = encoded["input_ids"][0].tolist()[:-num_vectors]
        attention_mask = encoded["attention_mask"][0].tolist()
        
        # Create labels for next token prediction
        # Shift input_ids by 1 for next token prediction
        labels = token_ids
        labels = torch.tensor(labels).to(dtype=torch.long)
        labels[torch.tensor(attention_mask) == 0] = -100
        # Pad each label tensor with number of vectors (-100) at the start
        n_vectors = labels.shape[0]  # assuming labels is 1D tensor [seq_len]
        pad = torch.full((n_vectors,), -100, dtype=labels.dtype)
        labels = torch.cat([pad, labels])
        
        return {
            "input_ids": encoded["input_ids"][0][:-num_vectors],
            "attention_mask": encoded["attention_mask"][0],
            "labels": labels,
        }


def collate_fn(batch):
    """
    Collate function for DataLoader.
    """
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

dataset = ConversationDataset(
    data_path=data_path,
    tokenizer=tokenizer,
)
dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
)

Loading dataset from HuggingFace: Bossologist/reddit-conversations-processed
Loaded 112594 examples from HuggingFace dataset


In [8]:
from transformers import get_linear_schedule_with_warmup
from accelerate import Accelerator

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Optimizer tracking {len(trainable_params)} parameter groups (trainable only)")
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=5e-5,
    weight_decay=0.01,
)

# Learning rate scheduler
num_epochs = 1
num_training_steps = len(dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
accelerator = Accelerator()
model, optimizer, dataloader, scheduler = accelerator.prepare(
    model, optimizer, dataloader, scheduler
)

Optimizer tracking 154 parameter groups (trainable only)


In [ ]:
from tqdm import tqdm

import os

def save_model(model, output_path, cur_dir):
    print("Saving final model...")
    save_dir = os.path.join(output_path, cur_dir)
    os.makedirs(save_dir, exist_ok=True)

    additional_components = {}
    additional_components["memory_proj"] = model.cross_proj.memory_proj.state_dict()
    additional_components["queries"] = model.cross_proj.queries.state_dict()
    additional_components["cross_attn"] = model.cross_proj.cross_attn.state_dict()
    torch.save(additional_components, os.path.join(output_path, f"{save_dir}_additional_components.pt"))
    model.qwen_model.save_pretrained(save_dir)

num_epochs = 1
global_step = 0
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in progress_bar:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = output.loss
        
        # Backward pass
        accelerator.backward(loss)
        accelerator.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        global_step += 1
        
        # Update progress bar
        progress_bar.set_postfix({"loss": loss.item(), "avg_loss": epoch_loss / global_step})
        save_model(model, "checkpoints/base_llm_qwen3-0.6b_lora", f"epoch_{epoch}")
        break
    
    # Epoch summary
    avg_epoch_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1} completed. Average loss: {avg_epoch_loss:.4f}")

Epoch 1/1: 100%|██████████| 14075/14075 [1:05:08<00:00,  3.60it/s, loss=3.61, avg_loss=3.76]


Epoch 1 completed. Average loss: 3.7627
Saving final model...
